In [1]:
import torch
import torch.nn as nn

## Transformer Encoder

Bu blokta:

* LayerNorm, token'ların özelliklerini normalize eder.

* Attention, token'lar arasında bilgi alışverişi sağlar.

* Residual bağlantılar önceki temsilleri korur.

* MLP, özellikleri dönüştürür.

* Çıkışın boyutu girişle aynı kalır.

In [2]:
# Pre-LayerNorm
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim=64, num_heads=8, mlp_ratio=4):
        super().__init__()

        self.norm1 = nn.LayerNorm(embed_dim)

        self.attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.norm2 = nn.LayerNorm(embed_dim)

        hidden_dim = embed_dim * mlp_ratio

        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim)
        )

    def forward(self, x):
        # Attention + residual
        z = self.norm1(x)
        attention_out, _ = self.attention(
            z, z, z, need_weights=False
        )
        x = x + attention_out

        # MLP + residual
        x = x + self.mlp(self.norm2(x))

        return x